# LLM From Scratch + Post-Training + Agentic RAG — Colab training
Runtime → Change runtime type → **T4 GPU**. Run cells top to bottom.

Add your Hugging Face write token as a Colab secret named `HF_TOKEN` (key icon in the left sidebar). Checkpoints sync to the Hub, so a disconnect only costs the steps since the last save.

In [ ]:
!git clone https://github.com/ahmedashrafk13/llm-from-scratch-agentic-rag.git
%cd llm-from-scratch-agentic-rag
!pip install -q transformers==4.46.3 datasets==3.1.0 peft==0.13.2 bitsandbytes accelerate sentence-transformers==3.3.1 faiss-cpu rank_bm25 regex gradio
from google.colab import userdata
from huggingface_hub import login, whoami
login(userdata.get('HF_TOKEN'))
HF_USER = whoami()['name']
HUB = f'{HF_USER}/llm-scratch-checkpoints'
print(HUB)

In [ ]:
!python -m pytest tests -q

## 1. Data + tokenizer (~25 min)
Also saves processed data to Drive so later sessions can skip this step.

In [ ]:
from google.colab import drive; drive.mount('/content/drive')
import os
D = '/content/drive/MyDrive/llm-scratch'
if os.path.exists(f'{D}/processed/train.bin'):
    !mkdir -p data/processed && cp -r {D}/processed/* data/processed/ && cp {D}/tokenizer.json tokenizer/
else:
    !python data/prepare_pretrain.py
    !python tokenizer/train_tokenizer.py
    !python data/tokenize_corpus.py
    !python data/build_sft.py
    !python evaluation/build_hallucination_set.py
    !mkdir -p {D}/processed && cp -r data/processed/* {D}/processed/ && cp tokenizer/tokenizer.json {D}/

## 2. Pre-training (~1 h on T4)
If the session disconnects: re-run the setup + data cells, then this one — `--resume` pulls the last checkpoint from the Hub.

In [ ]:
!python training/pretrain.py --batch 32 --accum 1 --resume --hub-repo {HUB}

## 3. SFT → DPO → reward model (~20 min)

In [ ]:
!python training/sft.py
!python training/dpo.py
!python training/reward_model.py
!cp -r checkpoints {D}/

## 4. QLoRA on Qwen2.5-1.5B (~35 min)

In [ ]:
!python training/qlora.py
!cp -r checkpoints {D}/

## 5. Evaluation + report (~1–1.5 h)

In [ ]:
!python evaluation/benchmarks.py
!python evaluation/retrieval_eval.py
!python evaluation/qa_eval.py
!python evaluation/make_report.py
!cp -r results {D}/

## 6. Upload trained models to the Hub (optional)

In [ ]:
from huggingface_hub import HfApi
api = HfApi()
api.create_repo(f'{HF_USER}/llm-scratch-37m', exist_ok=True)
api.upload_folder(folder_path='checkpoints', repo_id=f'{HF_USER}/llm-scratch-37m', ignore_patterns=['pretrain/*'])
print('done')